In [ ]:
# 1. Ensure the Gold database exists
spark.sql("CREATE DATABASE IF NOT EXISTS de_mini_project.gold")

# 2. Create the Gold Inventory Fact Table
# We join Inventory with Sales to identify "Slow-Moving" items
spark.sql("""
CREATE OR REPLACE TABLE de_mini_project.gold.fact_inventory AS
WITH latest_sales AS (
    -- Find the last time each product was sold at each store
    SELECT 
        product_id, 
        store_id, 
        MAX(date) as last_sale_date,
        SUM(CASE WHEN date >= date_sub(current_date(), 30) THEN sold_qty ELSE 0 END) as sales_last_30_days
    FROM de_mini_project.gold.fact_sales
    GROUP BY 1, 2
)
SELECT 
    i.sku_id as product_id,
    i.store_id,
    i.stocks_qty,
    i.last_audit_date,
    -- KPI #6: Out-of-Stock Flag
    CASE WHEN i.stocks_qty = 0 THEN 1 ELSE 0 END AS is_out_of_stock,
    -- KPI #10: Slow-Moving Flag (No sales in last 30 days)
    CASE WHEN COALESCE(s.sales_last_30_days, 0) = 0 THEN 1 ELSE 0 END AS is_slow_moving,
    s.last_sale_date
FROM de_mini_project.silver.inventory i
LEFT JOIN latest_sales s 
    ON i.sku_id = s.product_id 
    AND i.store_id = s.store_id
""")

# 3. Validation: Count how many items are currently out of stock
display(spark.sql("SELECT store_id, SUM(is_out_of_stock) as oos_count FROM de_mini_project.gold.fact_inventory GROUP BY 1"))